<a href="https://colab.research.google.com/github/DillonR03/Potential-Talents-Candidate-Ranking-Model/blob/main/Fine_Tuning_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ), "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
!git clone https://github.com/DillonR03/Potential-Talents-Candidate-Ranking-Model.git
%cd Potential-Talents-Candidate-Ranking-Model

Cloning into 'Potential-Talents-Candidate-Ranking-Model'...
remote: Enumerating objects: 15123, done.
remote: Total 15123 (delta 0), reused 0 (delta 0), pack-reused 15123 (from 2)
Receiving objects: 100% (15123/15123), 80.11 MiB | 12.78 MiB/s, done.
Resolving deltas: 100% (1516/1516), done.
Updating files: 100% (14796/14796), done.
/content/Potential-Talents-Candidate-Ranking-Model


In [ ]:
!ls

data  PotentialTalentsNotebook.ipynb  README.md


In [ ]:
!pip install -U transformers datasets peft trl accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Tokenizer loaded successfully")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully


In [ ]:
import pandas as pd

df = pd.read_excel(
    "data/Extended Dataset for Potential Talents 7.xlsx"
)

print(df.shape)
df.head()

(1285, 4)


,id,title,location,screening_score
0,1,innovative and driven professional seeking a r...,United States,100
1,2,ms applied data science student usc research a...,United States,100
2,3,computer science student seeking full-time sof...,United States,100
3,4,microsoft certified power bi data analyst mba ...,United States,100
4,5,graduate research assistant at uab masters in ...,United States,100


In [ ]:
pd.set_option("display.max_colwidth", 200)

display(df.head(10))

,id,title,location,screening_score
0,1,innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.,United States,100
1,2,ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025,United States,100
2,3,computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs,United States,100
3,4,microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-eri...,United States,100
4,5,graduate research assistant at uab masters in data science student at uab ex jio,United States,100
5,6,student at kennesaw state university,United States,100
6,7,data analyst business analyst python snowflake sql machine learning power bi tableau equipped with analytics driven by insights and passionate about impactful solutions.,United States,100
7,8,graduate research aide student at arizona state university,United States,100
8,9,data science nlp ai ml python sas quantum computing block chain sql,United States,100
9,10,data science machine learning artificial intelligence,United States,100


In [ ]:
for column in df.columns:
    print(f"\n===== {column} =====")
    print(df[column].head(5).tolist())


===== id =====
[1, 2, 3, 4, 5]

===== title =====
['innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.', 'ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025', 'computer science student seeking full-time software engineerdeveloper positions ai sql data visualization toolspython ssrs', 'microsoft certified power bi data analyst mba business analytics unt business intelligence engineer data scientist data engineer business analytics predictive analytics statistical analysis ex-ericsson', 'graduate research assistant at uab masters in data science student at uab ex jio']

===== location =====
['United States', 'United States', 'United States', 'United States', 'United States']

===== screening_score =====
[100, 100, 100, 100, 100]


In [ ]:
SYSTEM_PROMPT = """You are an expert recruitment analyst.\n\nEvaluate how well a candidate matches this recruitment search:\n\nTARGET SEARCH:\nAspiring human resources OR seeking human resources\n\nEvaluate the candidate using ONLY the information contained in their title.\n\nReturn ONLY an integer from 0 to 100 representing the candidate's screening score.\n"""

In [ ]:
df["text"] = df.apply(
    lambda x: f"{SYSTEM_PROMPT}\n### Human: Given the candidate's title: {x['title']}. What is the screening score? ### Assistant: {x['screening_score']}",
    axis=1,
)
print("Text column created with system prompt and title-based screening format.")

Text column created with system prompt and title-based screening format.


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

print(f"Dataset created: {dataset}")

Dataset created: Dataset({
    features: ['id', 'title', 'location', 'screening_score', 'text'],
    num_rows: 1285
})


In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Dataset split into train and test: {dataset}")

Dataset split into train and test: DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'location', 'screening_score', 'text'],
        num_rows: 1028
    })
    test: Dataset({
        features: ['id', 'title', 'location', 'screening_score', 'text'],
        num_rows: 257
    })
})


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load the base model with quantization config
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Set tokenizer padding token to be the same as the EOS token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

print("Base model and tokenizer loaded with quantization.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Base model and tokenizer loaded with quantization.


Next, we'll set up the LoRA configuration to enable efficient fine-tuning of the model.

In [ ]:
from peft import LoraConfig

# LoRA configuration
lora_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA configuration set.")

LoRA configuration set.


Now, we define the training arguments for our `SFTTrainer`.

In [ ]:
from transformers import TrainingArguments

# Training arguments
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=100,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    # warmup_ratio=0.03, # Removed as it causes TypeError
    # group_by_length=True, # Removed as it causes TypeError
    lr_scheduler_type="constant",
    report_to="tensorboard"
)

print("Training arguments defined.")

Training arguments defined.


Finally, we initialize the `SFTTrainer` and start the fine-tuning process.

In [ ]:
from trl import SFTTrainer

# Initialize SFTTrainer
sft_trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=lora_config,
    # dataset_text_field="text", # Removed as it causes TypeError
    # tokenizer=tokenizer, # Removed as it causes TypeError
    args=training_arguments,
    # packing=False, # Removed as it causes TypeError
    # max_seq_length=1024 # Removed as it causes TypeError
)

# Train the model
sft_trainer.train()

print("Model training started.")

Adding EOS to train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1028 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
100,1.287511
200,0.634621


Model training started.


In [ ]:
# Evaluate the model
metrics = sft_trainer.evaluate()
print("Evaluation Metrics:", metrics)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.634621,0.614084,257,0.696113,96463.000000,0.884695


Evaluation Metrics: {'eval_loss': 0.6140836477279663, 'eval_entropy': 0.696112643588673, 'eval_num_tokens': 96463.0, 'eval_mean_token_accuracy': 0.8846951506354592}


In [ ]:
# Save the fine-tuned model
output_merged_dir = "./results/final_merged_model"
sft_trainer.model.save_pretrained(output_merged_dir)
tokenizer.save_pretrained(output_merged_dir)
print(f"Fine-tuned model and tokenizer saved to {output_merged_dir}")

Fine-tuned model and tokenizer saved to ./results/final_merged_model
